# **YOLOv10 License Plate Detection Training**
This notebook is fully self-contained and fixes the path errors. Run all cells from top to bottom.

## **Step 01 & 02: Clone Repo and Install Packages**

In [ ]:
!git clone https://github.com/THU-MIG/yolov10.git
%cd yolov10
!pip install -q .
!pip install -q roboflow

## **Step 03: Download Pretrained Weights**

In [ ]:
import os
import urllib.request

weights_dir = os.path.join(os.getcwd(), 'weights')
os.makedirs(weights_dir, exist_ok=True)

url = "https://github.com/THU-MIG/yolov10/releases/download/v1.1/yolov10n.pt"
filepath = os.path.join(weights_dir, 'yolov10n.pt')
urllib.request.urlretrieve(url, filepath)
print(f"Downloaded: {filepath}")

## **Step 04: Download Dataset from Roboflow**
**Want more data?** Go to [Roboflow Universe](https://universe.roboflow.com), search for 'License Plate', click a dataset, and click **'Merge with my Dataset'**. Then, just change `version(2)` below to `version(3)` to download the combined Super-Dataset!

In [ ]:
from roboflow import Roboflow

# Using the updated API key provided
rf = Roboflow(api_key="YdiZ921aSbi8Qw0eErvl")
project = rf.workspace("moin").project("car_license_plates")

# If you merged new datasets on Roboflow, change this to version(3) or version(4)!
version = project.version(2)
dataset = version.download("yolov8")

# Store the exact path to data.yaml dynamically so it never errors out
dataset_yaml = f"{dataset.location}/data.yaml"
print("\nDataset ready! Configuration file at:", dataset_yaml)

## **Step 05: Train Custom YOLOv10 Model**

In [ ]:
!yolo task=detect mode=train epochs=120 batch=16 plots=True model='weights/yolov10n.pt' data='{dataset_yaml}'

## **Step 06: Download the Best Weights**
Once training completes, run this cell to securely download your `best.pt` file.

In [ ]:
import os
from google.colab import files

run_dirs = [d for d in os.listdir('runs/detect') if d.startswith('train')]
latest_run = max(run_dirs, key=lambda d: os.path.getmtime(os.path.join('runs/detect', d)))
best_pt_path = os.path.join('runs/detect', latest_run, 'weights', 'best.pt')

print(f"Downloading weights from: {best_pt_path}")
files.download(best_pt_path)